# BitNet b1.58 — YaRN Context Extension (4K → 8K)
**Local RTX 4090 run.** Goal: confirm 8K loss decreases over 200 steps.  
Model loaded from local disk. No HF download required.

---


## 1. Install dependencies

In [ ]:
# Run once. If already installed in your venv, skip.
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.46.3', 'accelerate', 'datasets',
    'safetensors', 'matplotlib'
], check=True)
print('Dependencies ready.')

## 2. Imports & config

In [ ]:
import os, math, time, types, random
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# ── Local paths ─────────────────────────────────────────────────────────────
# Adjust these if your folder layout differs
BITNET_ROOT   = Path(r'D:\Code\Bitnet')
MODEL_DIR     = BITNET_ROOT / 'models' / 'bitnet-hf'      # bf16 master weights
YARN_NPY      = BITNET_ROOT / 'yarn_inv_freq.npy'         # pre-computed inv_freq
OUTPUT_DIR    = BITNET_ROOT / 'yarn_checkpoints'          # where to save checkpoints
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert MODEL_DIR.exists(), f'Model not found: {MODEL_DIR}'
assert YARN_NPY.exists(),  f'yarn_inv_freq.npy not found: {YARN_NPY}'
print(f'Model dir : {MODEL_DIR}')
print(f'YaRN npy  : {YARN_NPY}')
print(f'Output dir: {OUTPUT_DIR}')

# ── RoPE / YaRN config (from forensic audit — do not change) ────────────────
ORIGINAL_MAX  = 4096
TARGET_MAX    = 8192
ROPE_THETA    = 500_000
HEAD_DIM      = 128
SCALE_ZONE    = (32, 42)   # bands to scale
SCALE_FACTOR  = 2.0

# ── Training config ──────────────────────────────────────────────────────────
LR            = 1e-5
TOTAL_STEPS   = 200
LOG_EVERY     = 10
SAVE_EVERY    = 100
BATCH_SIZE    = 1
GRAD_ACCUM    = 8          # effective batch = 8
MAX_SEQ_LEN   = 8192
USE_GRAD_CKPT = False      # 4090 has 24GB — not needed for 2B bf16. Set True if OOM.

DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE         = torch.bfloat16

assert DEVICE == 'cuda', 'No CUDA device found. Make sure PyTorch+CUDA is installed in this venv.'
print(f'Device : {DEVICE}')
print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

## 3. Load bf16 model from local disk

In [ ]:
import sys, os


CUSTOM_FILES = ['modeling_bitnet.py', 'configuration_bitnet.py']
SCRIPT_DIR   = Path(r'D:\Code\Bitnet')   # wherever you saved the two .py files

for fname in CUSTOM_FILES:
    src  = SCRIPT_DIR / fname
    dst  = MODEL_DIR  / fname
    if not dst.exists():
        import shutil
        shutil.copy(str(src), str(dst))
        print(f'Copied {fname} → {MODEL_DIR}')
    else:
        print(f'Already present: {fname}')

print('\nLoading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Loading model (bf16) from local disk...')
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=DTYPE,
    device_map='auto',
    low_cpu_mem_usage=True,
    local_files_only=True,
    trust_remote_code=True,
)
print(f'Loaded in {time.time() - t0:.1f}s')
print(f'Params : {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')
sub_norm_count = sum(1 for n, _ in model.named_parameters() if 'sub_norm' in n)
print(f'Sub_norm tensors : {sub_norm_count}  (expect 60)')

has_rotary = sum(1 for layer in model.model.layers if hasattr(layer.self_attn, 'rotary_emb'))
print(f'Layers with rotary_emb : {has_rotary}  (expect 30)')

assert sub_norm_count == 60, f'sub_norm count wrong: {sub_norm_count}'
assert has_rotary      == 30, f'rotary_emb count wrong: {has_rotary}'
print('\n✓ Model loaded and verified.')

## 4. YaRN RoPE patch — load pre-computed inv_freq and inject

In [ ]:
# Load the pre-computed YaRN inv_freq (already validated in forensic session)
yarn_inv_freq_np = np.load(str(YARN_NPY))   # shape (64,), float64
yarn_inv_freq_t  = torch.tensor(yarn_inv_freq_np, dtype=torch.float32).to(DEVICE)

print(f'yarn_inv_freq shape : {yarn_inv_freq_t.shape}')
print(f'yarn_inv_freq dtype : {yarn_inv_freq_t.dtype}')
print(f'yarn_inv_freq range : [{yarn_inv_freq_t.min().item():.6e}, {yarn_inv_freq_t.max().item():.6e}]')

# Quick sanity: band 32 wavelength should be ~2x the base wavelength
wl_32_base = (2 * math.pi) / (1.0 / (ROPE_THETA ** (64 / HEAD_DIM)))
wl_32_yarn = (2 * math.pi) / yarn_inv_freq_t[32].item()
ratio = wl_32_yarn / wl_32_base
assert abs(ratio - 2.0) < 0.001, f'Band 32 ratio wrong: {ratio:.4f} (expect 2.000)'
print(f'Band 32 wavelength — base: {wl_32_base:.1f}  yarn: {wl_32_yarn:.1f}  ratio: {ratio:.3f}  ✓')

# Inject into every attention layer
patched = 0
for layer in model.model.layers:
    attn = layer.self_attn
    if hasattr(attn, 'rotary_emb'):
        attn.rotary_emb.register_buffer(
            'inv_freq',
            yarn_inv_freq_t.clone(),
            persistent=True
        )
        patched += 1

# Update config so position index generation uses TARGET_MAX
model.config.max_position_embeddings = TARGET_MAX
if hasattr(model, 'generation_config') and model.generation_config is not None:
    model.generation_config.max_length = TARGET_MAX

print(f'Patched rotary_emb in {patched} layers  (expect 30)')

# Confirm band 0 and band 63 are untouched (paranoia check)
base_inv_freq_0  = 1.0 / (ROPE_THETA ** (0 / HEAD_DIM))   # = 1.0
base_inv_freq_63 = 1.0 / (ROPE_THETA ** (126 / HEAD_DIM))
yarn_0  = yarn_inv_freq_t[0].item()
yarn_63 = yarn_inv_freq_t[63].item()
print(f'Band  0: base={base_inv_freq_0:.6f}  yarn={yarn_0:.6f}  match={abs(yarn_0  - base_inv_freq_0)  < 1e-5}')
print(f'Band 63: base={base_inv_freq_63:.6e} yarn={yarn_63:.6e}  match={abs(yarn_63 - base_inv_freq_63) < 1e-9}')

## 5. Freeze strategy — layers 25-29 sub_norm (phase 1)

In [ ]:
HIGH_GAIN_FREEZE = list(range(25, 30))  # layers 25-29 (chaotic sub_norm region)

def set_frozen(freeze_high_gain: bool):
    for name, param in model.named_parameters():
        if any(f'layers.{i}.' in name for i in HIGH_GAIN_FREEZE) and 'sub_norm' in name:
            param.requires_grad = not freeze_high_gain
        else:
            param.requires_grad = True

    frozen    = sum(p.numel() for n, p in model.named_parameters() if not p.requires_grad)
    trainable = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad)
    print(f'Frozen   : {frozen / 1e6:.2f}M params')
    print(f'Trainable: {trainable / 1e6:.2f}M params')

print('Phase 1: freezing layers 25-29 sub_norm')
set_frozen(freeze_high_gain=True)

## 6. Dataset — PG19 long documents (streaming)

In [ ]:
print('Loading PG19 train split (streaming)...')
dataset = load_dataset('deepmind/pg19', split='train', streaming=True, trust_remote_code=True)

def tokenize_long_docs(dataset, tokenizer, min_tokens=6000, chunk_size=8192, n_chunks=500):
    """
    Stream PG19, skip docs shorter than min_tokens, chunk to exactly chunk_size tokens.
    Returns list of LongTensors ready for training.
    """
    chunks = []
    buffer = []

    for doc in dataset:
        if len(chunks) >= n_chunks:
            break
        ids = tokenizer(doc['text'], add_special_tokens=False)['input_ids']
        if len(ids) < min_tokens:
            continue
        buffer.extend(ids)
        while len(buffer) >= chunk_size:
            chunks.append(torch.tensor(buffer[:chunk_size], dtype=torch.long))
            buffer = buffer[chunk_size:]

    print(f'Built {len(chunks)} chunks of {chunk_size} tokens')
    return chunks

chunks = tokenize_long_docs(dataset, tokenizer, n_chunks=500)
print(f'Dataset ready. Total tokens: {len(chunks) * MAX_SEQ_LEN / 1e6:.1f}M')

## 7. Baseline — PPL at 4K and 8K (before training)

In [ ]:
@torch.no_grad()
def compute_ppl(model, chunks, max_len, n_samples=20):
    """Compute perplexity on first n_samples chunks truncated to max_len."""
    model.eval()
    total_loss   = 0.0
    total_tokens = 0

    for chunk in chunks[:n_samples]:
        ids = chunk[:max_len].unsqueeze(0).to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=DTYPE):
            out = model(ids, labels=ids)
        total_loss   += out.loss.item() * (ids.shape[1] - 1)
        total_tokens += ids.shape[1] - 1

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss), avg_loss

print('Computing baseline PPL at 4K tokens...')
ppl_4k_base, loss_4k_base = compute_ppl(model, chunks, max_len=4096)
print(f'  PPL@4K  = {ppl_4k_base:.3f}  (loss={loss_4k_base:.4f})')

print('Computing baseline PPL at 8K tokens...')
ppl_8k_base, loss_8k_base = compute_ppl(model, chunks, max_len=8192)
print(f'  PPL@8K  = {ppl_8k_base:.3f}  (loss={loss_8k_base:.4f})')

print(f'\nBaseline degradation at 8K: {ppl_8k_base - ppl_4k_base:+.3f} PPL points')
print('(expect high — model has not been trained on 8K context)')

## 8. Training loop — 200 steps

In [ ]:
from torch.optim import AdamW

if USE_GRAD_CKPT:
    model.gradient_checkpointing_enable()
    print('Gradient checkpointing: ON')
else:
    print('Gradient checkpointing: OFF  (24GB VRAM sufficient for 2B bf16)')

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=0.01,
)

model.train()

log = {'step': [], 'loss_4k': [], 'loss_8k': [], 'phase': []}

print(f'Starting training — {TOTAL_STEPS} steps, lr={LR}')
print(f'Phase 1 (steps   0–99): layers 25-29 sub_norm FROZEN')
print(f'Phase 2 (steps 100–199): layers 25-29 sub_norm UNFROZEN')
print('─' * 65)

optimizer.zero_grad()
t_start = time.time()

for step in range(TOTAL_STEPS):

    # ── Phase transition at step 100 ──────────────────────────────────────
    if step == 100:
        print('\n→ Phase 2: unfreezing layers 25-29 sub_norm')
        set_frozen(freeze_high_gain=False)
        optimizer = AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR * 0.5,   # halve lr for cautious unfreeze
            weight_decay=0.01,
        )
        optimizer.zero_grad()

    # ── Sample a random 8K chunk ──────────────────────────────────────────
    chunk = random.choice(chunks).to(DEVICE)
    ids   = chunk.unsqueeze(0)   # (1, 8192)

    # ── Forward + loss ────────────────────────────────────────────────────
    with torch.autocast(device_type='cuda', dtype=DTYPE):
        out = model(ids, labels=ids)
    loss = out.loss / GRAD_ACCUM
    loss.backward()

    # ── Gradient accumulation ─────────────────────────────────────────────
    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

    # ── Logging ───────────────────────────────────────────────────────────
    if step % LOG_EVERY == 0:
        elapsed    = time.time() - t_start
        steps_left = TOTAL_STEPS - step
        eta        = (elapsed / max(step, 1)) * steps_left

        model.eval()
        with torch.no_grad():
            _, l4k = compute_ppl(model, chunks, max_len=4096, n_samples=5)
            _, l8k = compute_ppl(model, chunks, max_len=8192, n_samples=5)
        model.train()

        phase = 1 if step < 100 else 2
        log['step'].append(step)
        log['loss_4k'].append(l4k)
        log['loss_8k'].append(l8k)
        log['phase'].append(phase)

        print(f'Step {step:>3} | loss@4K={l4k:.4f} | loss@8K={l8k:.4f} | '
              f'gap={l8k - l4k:+.4f} | phase={phase} | ETA={eta / 60:.1f}min')

    # ── Checkpoint ────────────────────────────────────────────────────────
    if (step + 1) % SAVE_EVERY == 0:
        ckpt_path = OUTPUT_DIR / f'step_{step + 1:04d}'
        model.save_pretrained(str(ckpt_path))
        tokenizer.save_pretrained(str(ckpt_path))
        print(f'  → Checkpoint saved: {ckpt_path}')

print('\nTraining complete.')

## 9. Post-training evaluation

In [ ]:
model.eval()

print('Post-training PPL (20 samples each):')
ppl_4k_after, loss_4k_after = compute_ppl(model, chunks, max_len=4096, n_samples=20)
ppl_8k_after, loss_8k_after = compute_ppl(model, chunks, max_len=8192, n_samples=20)

print(f'  PPL@4K  : {ppl_4k_base:.3f} → {ppl_4k_after:.3f}  (Δ {ppl_4k_after - ppl_4k_base:+.3f})')
print(f'  PPL@8K  : {ppl_8k_base:.3f} → {ppl_8k_after:.3f}  (Δ {ppl_8k_after - ppl_8k_base:+.3f})')
print()

delta_4k = ppl_4k_after - ppl_4k_base
delta_8k = ppl_8k_after - ppl_8k_base

if delta_8k < -1.0 and delta_4k < 1.0:
    print('✓ SUCCESS: 8K PPL improved, 4K PPL preserved. Proceed to full 1000-step run.')
elif delta_8k < 0:
    print('~ PARTIAL: 8K PPL improving but slowly. More steps needed.')
elif delta_4k > 2.0:
    print('✗ WARNING: 4K PPL degraded significantly. Check YaRN patch and lr.')
else:
    print('? INCONCLUSIVE: Need more steps.')

## 10. Loss curve

In [ ]:
import matplotlib.pyplot as plt

steps_log = log['step']
loss_4k   = log['loss_4k']
loss_8k   = log['loss_8k']
phases    = log['phase']

fig, ax = plt.subplots(figsize=(12, 5))

phase2_start = next((s for s, p in zip(steps_log, phases) if p == 2), None)
if phase2_start is not None:
    ax.axvspan(phase2_start, steps_log[-1], alpha=0.08, color='orange', label='Phase 2 (unfrozen)')

ax.plot(steps_log, loss_4k, 'b-o', markersize=4, label='Loss @ 4K tokens', linewidth=2)
ax.plot(steps_log, loss_8k, 'r-o', markersize=4, label='Loss @ 8K tokens', linewidth=2)
ax.axhline(loss_4k_base, color='b', linestyle='--', alpha=0.4, label=f'Baseline 4K ({loss_4k_base:.4f})')
ax.axhline(loss_8k_base, color='r', linestyle='--', alpha=0.4, label=f'Baseline 8K ({loss_8k_base:.4f})')

ax.set_xlabel('Training Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('BitNet b1.58 YaRN Context Extension — 4K vs 8K Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

curve_path = OUTPUT_DIR / 'loss_curve.png'
plt.savefig(str(curve_path), dpi=150)
plt.show()
print(f'Saved: {curve_path}')

## 11. Save final model

In [ ]:
final_path = OUTPUT_DIR / 'final_200steps'
model.save_pretrained(str(final_path))
tokenizer.save_pretrained(str(final_path))

# Also save the YaRN inv_freq alongside the checkpoint for reproducibility
import shutil
shutil.copy(str(YARN_NPY), str(final_path / 'yarn_inv_freq.npy'))

print(f'Final model saved to: {final_path}')
print(f'Loss curve saved to : {OUTPUT_DIR / "loss_curve.png"}')
print()
print('Done. Check loss_curve.png to decide whether to proceed to 1000-step run.')